In [ ]:
from google.colab import drive
import os

# 1. 구글 드라이브 마운트 (팝업창에서 '허용'을 눌러주세요)
drive.mount('/content/drive')

# 2. 경로 설정
zip_path = '/content/drive/MyDrive/fine_tuning_dataset_v3.1.zip'
target_dir = '/content'  # 코랩 최상위에 풀면 알아서 v3.1 폴더가 생성됩니다.

print(f"\n구글 드라이브의 '{os.path.basename(zip_path)}' 파일을 코랩 로컬로 푸는 중입니다...")
print("잠시만 기다려주세요! \n")

if os.path.exists(zip_path):
    # 3. 초고속 압축 해제 (-q: 조용히, -o: 덮어쓰기, -d: 지정 경로에 풀기)
    !unzip -q -o "{zip_path}" -d "{target_dir}"

    # 4. 압축이 잘 풀렸는지 최종 확인
    check_path = '/content/fine_tuning_dataset_v3.1'
    if os.path.exists(check_path):
        print("-" * 50)
        print("압축 해제 완료! 데이터셋이 코랩 로컬에 준비되었습니다.")
        print(f"데이터 위치: {check_path}")
        print("왼쪽 파일 탐색기를 새로고침해서 'images'와 'labels'가 잘 들어왔는지 확인해 보세요!")
    else:
        print("\n압축은 풀렸으나 예상된 폴더를 찾지 못했습니다. 파일 탐색기를 새로고침해서 폴더 이름을 확인해 보세요.")
else:
    print("\n오류: 드라이브에서 zip 파일을 찾을 수 없습니다.")
    print("구글 드라이브 최상위 경로에 'fine_tuning_dataset_v3.1.zip'이 있는지 확인해 주세요.")

Mounted at /content/drive

⏳ 구글 드라이브의 'fine_tuning_dataset_v3.1.zip' 파일을 코랩 로컬로 푸는 중입니다...
데이터가 많아 1~3분 정도 소요될 수 있습니다. 잠시만 기다려주세요! ☕

--------------------------------------------------
🎉 압축 해제 완료! 데이터셋이 코랩 로컬에 완벽하게 준비되었습니다.
📁 데이터 위치: /content/fine_tuning_dataset_v3.1
왼쪽 파일 탐색기(📁)를 새로고침(🔄)해서 'images'와 'labels'가 잘 들어왔는지 확인해 보세요!


In [ ]:
import yaml
import os

# v3.1 경로에 맞게 yaml 파일 생성
dataset_path = '/content/fine_tuning_dataset_v3.1'
yaml_path = os.path.join(dataset_path, 'data.yaml')

data = {
    'train': os.path.join(dataset_path, 'images/train'),
    'val': os.path.join(dataset_path, 'images/val'),
    'nc': 2,
    'names': ['YES_Helmet', 'NO_Helmet']
}

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(f"'{yaml_path}' 생성 완료!")

✅ '/content/fine_tuning_dataset_v3.1/data.yaml' 생성 완료!


In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# 1. 이전 학습에서 가장 성능이 좋았던 모델 로드
# (반드시 기존에 학습된 best.pt 파일이 준비되어 있어야 합니다)
model = YOLO('best.pt')

print("v3.1 오답 노트(모자/안경) 집중 파인튜닝을 시작합니다...\n")

# 2. 파인튜닝 시작
results = model.train(
    data='/content/fine_tuning_dataset_v3.1/data.yaml',
    optimizer='MuSGD',
    epochs=50,
    patience=10,
    imgsz=640,
    batch=128,
    workers=8,

    # 기존(0.02)은 처음부터 배울 때 좋지만, 지금은 기존 지식을 망가뜨리지 않아야 하므로 낮춤
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=1.0,         # 초반 학습률 워밍업 기간 단축 (이미 학습된 모델이므로)

    momentum=0.937,
    weight_decay=0.0005,
    freeze=0,                  # 전체 레이어를 미세하게 업데이트

    # 분류 손실 가중치 증가
    # 모자(NO_Helmet)를 안전모(YES_Helmet)로 잘못 분류했을 때 모델에게 주는 '고통(페널티)'을 늘립니다.
    cls=3.0,                   # 기존 2.5 -> 3.0 으로 상향

    save_period=5,
    mosaic=1.0,

    # Mixup 소폭 감소
    # Mixup은 이미지를 반투명하게 겹치는 기법인데, 너무 겹치면 모자와 안전모의 미세한 질감 차이가 뭉개질 수 있습니다.
    mixup=0.05,                # 기존 0.1 -> 0.05 로 하향
)

print("\n파인튜닝 완료")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 v3.1 오답 노트(모자/안경) 집중 파인튜닝을 시작합니다...

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fine_tuning_dataset_v3.1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction

In [4]:
from ultralytics import YOLO
import cv2

model = YOLO('/content/best.pt')

# 영상 경로 설정
video_path = '/content/cctv.mp4'

# 추론 실행 및 결과 저장
# conf=0.3: 신뢰도 임계값, 원하시는 수치로 조정 가능합니다.
# save=True: 결과를 영상으로 저장합니다.
results = model.predict(
    source=video_path,
    conf=0.3,
    save=True,
    imgsz=1024,
    device=0,           # GPU 사용
    stream=True         # 대용량 영상 처리를 위한 스트리밍 방식
)

# 결과 출력
# stream=True를 사용하면 generator 객체가 반환되므로 루프를 돌려야 합니다.
for r in results:
    # 각 프레임별로 추론 결과를 처리하거나 시각화할 수 있습니다.
    # r.plot()을 사용하면 바운딩 박스가 그려진 이미지가 생성됩니다.
    annotated_frame = r.plot()

    # 화면 표시를 원하시면 cv2.imshow를 쓰지만, 코랩에서는
    # 자동으로 save=True 설정에 의해 파일로 저장됩니다.
    pass

print("영상 추론 및 결과 저장 완료!")


video 1/1 (frame 1/102) /content/cctv.mp4: 576x1024 1 YES_Helmet, 124.9ms
video 1/1 (frame 2/102) /content/cctv.mp4: 576x1024 1 YES_Helmet, 11.4ms
video 1/1 (frame 3/102) /content/cctv.mp4: 576x1024 1 YES_Helmet, 11.1ms
video 1/1 (frame 4/102) /content/cctv.mp4: 576x1024 (no detections), 11.3ms
video 1/1 (frame 5/102) /content/cctv.mp4: 576x1024 2 YES_Helmets, 11.1ms
video 1/1 (frame 6/102) /content/cctv.mp4: 576x1024 (no detections), 10.9ms
video 1/1 (frame 7/102) /content/cctv.mp4: 576x1024 (no detections), 10.8ms
video 1/1 (frame 8/102) /content/cctv.mp4: 576x1024 1 YES_Helmet, 10.8ms
video 1/1 (frame 9/102) /content/cctv.mp4: 576x1024 (no detections), 10.8ms
video 1/1 (frame 10/102) /content/cctv.mp4: 576x1024 (no detections), 10.9ms
video 1/1 (frame 11/102) /content/cctv.mp4: 576x1024 2 YES_Helmets, 10.7ms
video 1/1 (frame 12/102) /content/cctv.mp4: 576x1024 1 YES_Helmet, 10.7ms
video 1/1 (frame 13/102) /content/cctv.mp4: 576x1024 (no detections), 10.6ms
video 1/1 (frame 14/102) 